**Pseudocódigo del algoritmo de control:**

In [ ]:
INICIO
  Definir P_ref = 2.5 bar
  Inicializar Kp = 1.0, Ki = 0.1, Kd = 0.05
  Inicializar error_prev = 0, integral = 0

  Bucle principal:
    Leer P_meas desde sensor de presión
    error = P_ref - P_meas
    integral = integral + error * dt
    derivada = (error - error_prev) / dt
    control = Kp * error + Ki * integral + Kd * derivada
    posicion_valvula = map(control, -100, 100, 0, 180)
    Mover servomotor MG996R a posicion_valvula

    Si |error| > 0.5 bar durante más de 5 segundos:
        Enviar alerta de “fuga o baja presión”
        Ajustar válvula al 80% de cierre preventivo
    FinSi

    error_prev = error
    Esperar 200 ms
  FinBucle
FIN


**Pseudocódigo del algoritmo de percepción:**

In [ ]:
INICIO
  Definir umbral_presion_baja = 1.0 bar
  Definir umbral_presion_alta = 4.0 bar
  Definir umbral_flujo_min = 0.5 L/min

  Bucle principal:
    Leer presion_bruta desde sensor de presión
    Leer flujo_bruto desde sensor de flujo

    presion_filtrada = filtro_promedio(presion_bruta)
    flujo_filtrado = filtro_promedio(flujo_bruto)

    Si flujo_filtrado > umbral_flujo_min Y presion_filtrada < umbral_presion_baja:
        Estado = “Fuga detectada”
        Cerrar válvula parcialmente (70%)
        Enviar alerta WiFi: “Posible fuga detectada”
    Sino si flujo_filtrado < umbral_flujo_min Y presion_filtrada > umbral_presion_alta:
        Estado = “Obstrucción detectada”
        Abrir válvula gradualmente (40%)
        Enviar alerta WiFi: “Posible bloqueo en línea hidráulica”
    Sino:
        Estado = “Operación normal”
    FinSi

    Esperar 200 ms
  FinBucle
FIN


Pseudocódigo de integración de ML:

In [ ]:
INICIO
  Cargar modelo entrenado “fugas_model.tflite”
  Inicializar sensores de flujo y presión
  Mientras (sistema activo):
      Leer presion_filtrada y flujo_filtrado
      Crear vector_entrada = [presion_filtrada, flujo_filtrado]
      resultado = modelo.predict(vector_entrada)

      Si resultado == “fuga”:
          Enviar alerta: “Fuga detectada - ML”
          Cerrar válvula al 80%
      Sino si resultado == “bloqueo”:
          Enviar alerta: “Obstrucción detectada - ML”
          Abrir válvula parcialmente
      Sino:
          Mantener operación normal
      FinSi
      Esperar 200 ms
  FinMientras
FIN
